# LNN 多周期融合模型 — TPU 训练 (Colab)

在 Google Colab TPU (v5e-1 / v2-8) 上训练液态神经网络加密货币涨跌预测模型。

**使用前注意：**
- **关键步骤:** 运行时 → 更改运行时类型 → **TPU** （不是 GPU/None）
- 本 notebook 会从 GitHub 拉取代码，训练完成后将最佳模型保存到 Google Drive
- 训练耗时约 2~3 小时（300 epochs）
- 网络请求获取交易所数据，请确保网络畅通

---
## 1. 安装依赖与挂载 Drive

In [ ]:
# 挂载 Google Drive（保存模型和 checkpoints）
from google.colab import drive
drive.mount('/content/drive')

# 创建项目目录
import os
PROJECT_DIR = '/content/drive/MyDrive/btclnn'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'项目目录: {PROJECT_DIR}')

In [ ]:
# TPU 诊断：检查加速器类型是否正确
import os
tpu_type = os.environ.get('COLAB_TPU_ACCELERATOR')
print(f'COLAB_TPU_ACCELERATOR = {tpu_type}')

if tpu_type is None:
    print()
    print('=' * 60)
    print('  ❌ 未检测到 TPU 加速器！')
    print('  请执行: 运行时 → 更改运行时类型 → TPU')
    print('  然后: 运行时 → 恢复出厂设置 → 从头重新运行')
    print('=' * 60)
elif tpu_type:
    print(f'  ✅ TPU 类型: {tpu_type}')

# 检查 /dev/vfio 状态（TPU 设备文件）
import subprocess
result = subprocess.run(['ls', '-la', '/dev/vfio/'], capture_output=True, text=True)
print(f'/dev/vfio/: {result.stdout.strip() if result.returncode == 0 else "NOT FOUND"}')
print()

# 尝试简单加载 torch_xla 看能否工作
import sys
try:
    import torch_xla
    print(f'torch_xla 版本: {torch_xla.__version__}')
    try:
        device = torch_xla.device() if hasattr(torch_xla, 'device') else None
        print(f'TPU 设备: {device}')
    except RuntimeError as e:
        print(f'  ⚠️ TPU 初始化失败: {e}')
        print(f'  >>> 修复: 运行时 → 恢复出厂设置，然后全部重新运行 <<<')
except ImportError:
    print(f'torch_xla 未安装')

print(f'Python 版本: {sys.version}')

In [ ]:
# 安装 torch_xla (仅在缺失时安装)
import sys
try:
    import torch_xla
    print(f'torch_xla 已安装: {torch_xla.__version__}')
except ImportError:
    print('安装 torch_xla for Colab TPU...')
    !pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

import torch
print(f'PyTorch 版本: {torch.__version__}')
os.environ['PJRT_DEVICE'] = 'TPU'

In [ ]:
# 克隆项目代码（替换为你的仓库地址）
REPO_URL = 'https://github.com/yourusername/btclnn.git'  # ← 修改此处
LOCAL_DIR = '/content/btclnn'

if os.path.exists(LOCAL_DIR):
    print('代码已存在，同步更新...')
    %cd {LOCAL_DIR}
    !git pull
else:
    print('克隆代码仓库...')
    !git clone {REPO_URL} {LOCAL_DIR}
    %cd {LOCAL_DIR}

print(f'当前目录: {os.getcwd()}')
print(f'文件列表:')
!ls -la

---
## 2. 配置 TPU 训练参数

将 checkpoint 和最终模型保存到 Google Drive，避免 Colab 断开后丢失。

In [ ]:
# 创建 Drive 上的 checkpoint 目录
DRIVE_CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

# 创建本地符号链接，使代码能直接保存到 Drive
LOCAL_CKPT = os.path.join(LOCAL_DIR, 'checkpoints')
if not os.path.exists(LOCAL_CKPT):
    os.symlink(DRIVE_CHECKPOINT_DIR, LOCAL_CKPT)
    print(f'创建符号链接: {LOCAL_CKPT} → {DRIVE_CHECKPOINT_DIR}')

# 也可直接软链接 data 缓存目录
DRIVE_DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
LOCAL_DATA = os.path.join(LOCAL_DIR, 'data')
if not os.path.exists(LOCAL_DATA):
    import shutil
    if os.path.isdir(LOCAL_DATA) and not os.path.islink(LOCAL_DATA):
        shutil.rmtree(LOCAL_DATA)
    os.symlink(DRIVE_DATA_DIR, LOCAL_DATA)
    print(f'创建符号链接: {LOCAL_DATA} → {DRIVE_DATA_DIR}')

print('\n目录结构:')
!ls -la {LOCAL_DIR}/checkpoints
!ls -la {LOCAL_DIR}/data

In [ ]:
# 安装项目其余依赖
!pip install -r tpu_train/requirements_tpu.txt

import numpy as np
print(f'numpy 版本: {np.__version__}')

---
## 3. 启动 TPU 训练

`tpu_train.py` 使用 `_TPUCompat` 兼容层自动适配 torch_xla 版本。

> **如果看到 "TPU 设备忙" 警告：**
>  1. 不要继续运行训练单元格
>  2. 执行: 运行时 → 恢复出厂设置 (Factory reset runtime)
>  3. 运行时 → 更改运行时类型 → **TPU**
>  4. 从**第 1 个单元格**开始重新执行

> **注意:** 训练前几分钟 XLA 编译计算图，之后每个 epoch 会很快。

In [ ]:
%cd {LOCAL_DIR}
!python tpu_train/tpu_train.py

---
## 4. 查看结果

In [ ]:
print('=' * 60)
print('训练完成! 模型文件:')
print('=' * 60)

import glob
ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints')
for f in sorted(glob.glob(os.path.join(ckpt_dir, '*'))):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f'  {os.path.basename(f):<40} {size_mb:.1f} MB')

---
## 5. 断点续训

如果运行时断开或 TPU 超时限制，重新打开 notebook 后：
1. 保持 Drive 挂载不变
2. 代码会从 GitHub 重新拉取
3. `checkpoints/` 符号链接到 Drive，已有模型不会丢失
4. 训练脚本的 `_load_fallback()` 会自动加载 Drive 上已有的最佳模型继续训练

修改 `tpu_config.py` 的 `MAX_TRAIN_SECONDS = 3600`（1 小时）可控制单次训练时长。